This is a test of the new data / background preparation process.

Let's start by downloading some test data

In [1]:
from gdt.missions.fermi.gbm.finders import TriggerFinder

finder = TriggerFinder('170817529')
tte_files = finder.get_tte(".") # Note: uses latest API from https://github.com/USRA-STI

Output()

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

Output()

Let's also define some search settings. These are things that could go into a settings class.

In [2]:
import numpy as np
from gdt.missions.fermi.gbm.detectors import GbmDetectors

nai_edges = np.array([0, 8, 20, 33, 51, 85, 106, 127, 128])
bgo_edges = np.array([0, 8, 21, 40, 65, 90, 112, 124, 128])

settings = {
    'win_width': 60,
    'min_loglr': 5,
    'min_dur': 0.064, 'max_dur': 8.192,
    'min_step': 0.064,'num_steps': 8,
    'detectors':
        {det.name: {'channel_edges': nai_edges, 'search_channels': [1, 2, 3, 4, 5, 6]} for det in GbmDetectors.nai()} |
        {det.name: {'channel_edges': bgo_edges, 'search_channels': [0, 1, 2, 3, 4, 5, 6, 7]} for det in GbmDetectors.bgo()},
}

# these items could be properties / mapping methods for a settings class
detectors = list(settings['detectors'].keys())
channel_edges = {det: settings['detectors'][det]['channel_edges'] for det in detectors}
time_range = np.array([-1, 1]) * max([0.5 * settings['win_width'] + settings['max_dur'] + 1.024, 30])
bkgd_range = [-40, 40]
phaii_resolution = settings['min_dur']

Now let's open the TTE data and update its trigtime to the time we want to search. This way all future times can be computed relative to this time of interest. We'll probably want a better way to handle this in the future using astropy.time objects.

During this step we'll also rebin the energy of the TTE files so that all future data / background formats that derive from it can inherit the same energy binning.

Store TTE data in a DataCollection for easier manipulation during future steps.

In [3]:
from rich.progress import track
from gdt.core.collection import DataCollection
from gdt.core.binning.binned import rebin_by_edge_index
from gdt.missions.fermi.gbm.tte import GbmTte

from data import update_tte_trigtime

t0 = 524666469.446
tte_data = []
for det in track(detectors, description="Opening TTE files"):
    path = f"data/gbm/524666469.429/glg_tte_{det}_170817_12z_v00.fit.gz"
    tte = update_tte_trigtime(GbmTte.open(path), t0)
    tte = tte.rebin_energy(rebin_by_edge_index, channel_edges[det])
    tte_data.append(tte)

ttes = DataCollection.from_list(tte_data, names=detectors)

Output()

Now let's pre-bin the TTE data in time using the minimum step resolution from the search settings. This will allow us to quickly sum time bins when performing the search itself.

In [4]:
from gdt.core.binning.unbinned import bin_by_time

phaiis = DataCollection.from_list(
    ttes.to_phaii(bin_by_time, phaii_resolution, time_ref=0, time_range=time_range),
    names=detectors)

Next we'll pass the Phaii data to a **PhaiiCountMatrix** class. The goal of this class is to act as an interface between the underlying data format (phaii) and the format needed by the likelihood method of the search (1D matrix of counts for all detectors and energy bins)

Note: this could probably be consolidated with the previous step.

In [5]:
from data import PhaiiCountMatrix

data = PhaiiCountMatrix(phaiis)
counts, exposure = data.counts(0, 1.024)
print("counts", counts)
print("exposure", exposure)

counts [ 72. 324. 242. 133. 169.  39.  27.  97.  71. 314. 226. 149. 135.  30.
  32.  76.  63. 283. 202. 155. 164.  39.  71.  38.  80. 391. 244. 158.
 151.  32.  22.  65.  61. 418. 269. 148. 131.  34.  44.  42.  82. 393.
 245. 165. 154.  45.  79.  32.  60. 295. 199. 152. 142.  34.  84.  26.
  81. 365. 217. 164. 149.  23.  38.  52.  63. 326. 220. 154. 152.  32.
  62.  52.  45. 254. 203. 165. 164.  43. 114.  15.  68. 132. 185. 136.
 141.  25.  73.  43.  35. 165. 128. 141. 151.  30.  35.  83. 477. 292.
 386. 136.  42.  27.  23. 109. 370. 319. 323. 140.  25.  28.  19. 117.]
exposure [1.0204144 1.0207518 1.0210798 1.0205472 1.020707  1.0206562 1.0212284
 1.0207838 1.0208566 1.0212812 1.021594  1.021389  1.0193142 1.0196476]


Next we'll fit the background using a polynomial fit applied to phaii data. This can easily be swapped for a NaisePoisson sliding window fit to TTE data.

In [6]:
from gdt.core.background.fitter import BackgroundFitter
from gdt.core.background.binned import Polynomial

backfitters = DataCollection.from_list(
    [BackgroundFitter.from_phaii(phaii, Polynomial, time_ranges=[bkgd_range]) for phaii in phaiis],
    names=detectors)
backfitters.fit(order=1)

Next we'll pass the fitters to **BackgroundRatesMatrix**, which is similar to **PhaiiCountMatrix** in that it prepares background rates in the format needed by the search.

Note: passing fitters to BackgroundRatesMatrix means that background generation is agnostic to the **method**. Users are free to apply whatever fit methods they want. For example, they may want to manually tune the background applied to each detector under complicated background scenarios.

In [7]:
from background import BackgroundRatesMatrix

background = BackgroundRatesMatrix(backfitters, time_range=bkgd_range)
rates, good = background.rates(0.512)
print("rates", rates)
print("good", good)

rates [ 69.23200098 301.21406401 198.91427801 156.6159865  148.46326225
  37.12347254  27.85430433  90.25801641  68.12019971 339.15057108
 226.58115886 158.9791138  141.15216837  35.11688432  40.30179227
  77.20532702  62.19048049 297.91996266 210.34627564 158.36967657
 149.50203114  38.57840889  69.85967179  46.70491255  85.12438339
 392.13872193 245.169549   169.61791003 134.05076544  31.5999261
  23.90714836  66.94048133  69.55174624 399.91845121 252.52215945
 168.98451943 133.30945434  31.49313961  53.43128119  45.3042406
  87.9654172  398.01969269 247.77073862 162.91325058 149.79424286
  36.54916241  69.0245233   27.96542394  68.43683767 289.83694102
 185.61571252 148.72016348 134.18322865  32.25981843  74.11075969
  21.0060798   85.69956445 361.30248887 224.52328794 159.65502715
 133.90407153  32.85440228  36.46312355  53.93251794  66.96242387
 328.60093681 205.72867459 155.68795142 133.29775443  34.01163732
  61.33717303  50.86294884  53.33209551 258.45423081 189.79195297
 152.2